In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from egh_vlm.hallu_dataset import load_hallu_dataset
from egh_vlm.extract_features import batch_extract_features
from egh_vlm.utils import ModelBundle, load_phd_dataset

## Analysis

In [ ]:
dataset_path = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
img_folder_path = f'{prefix_path}/data/phd/images'
features_masked_image_path = f'{prefix_path}/data/phd/features_masked_image.pt'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype='auto',
    device_map=device
)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 768)
model_bundle = ModelBundle(model, processor, device)

In [ ]:
dataset = load_phd_dataset(
    dataset_path=dataset_path, 
    img_folder_path=img_folder_path,
    sample_size=100)

features = load_hallu_dataset(features_masked_image_path) if os.path.isfile(features_masked_image_path) else {}
print(f'Length of processed features_masked_image: {len(features)}')

In [ ]:
features = batch_extract_features(
    dataset=dataset,
    model_bundle=model_bundle,
    processed_features=features,
    mask_mode='image',
    targeted_layer=-1,
    save_path=features_masked_image_path
)
if len(features) > 0:
    print('Shape of embedding:', features.embs[0].shape)
    print('Shape of gradient:', features.grads[0].shape)